# Phase 1: Translate Questions

Translates all natural language questions in Spider's dev/train JSON files
from English to Indonesian using Google Translate.

**Requires**: Phase 0 schema dict (uploaded as Kaggle dataset)

**Estimated API requests**: ~9,700 (dev=1,034 + train_spider=7,000 + train_others=1,659)

In [ ]:
!pip install -q googletrans==4.0.2 tqdm tenacity nest_asyncio

In [ ]:
import os, sys, zipfile, urllib.request

# === CONFIGURATION ===
SPIDER_ZIP_URL = 'https://github.com/mfazrinizar/mfazrinizar/releases/download/1.0.0/spider.zip'
WORK_DIR = '/kaggle/working'
SPIDER_DIR = os.path.join(WORK_DIR, 'spider')
OUTPUT_DIR = os.path.join(WORK_DIR, 'translated_questions')
SRC_DIR = '/kaggle/input/idspider1-src-new'  # Upload src_new/ as Kaggle dataset

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Download and extract Spider dataset
zip_path = os.path.join(WORK_DIR, 'spider.zip')
if not os.path.exists(SPIDER_DIR):
    print('Downloading spider.zip...')
    urllib.request.urlretrieve(SPIDER_ZIP_URL, zip_path)
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    os.remove(zip_path)
    print(f'Spider dataset at: {SPIDER_DIR}')
else:
    print('Spider dataset already exists')

print('Contents:', os.listdir(SPIDER_DIR))

In [ ]:
# Add src_new to path
sys.path.insert(0, os.path.dirname(SRC_DIR))

from pathlib import Path
from src_new.translate_questions import run_phase1

run_phase1(
    spider_dir=Path(SPIDER_DIR),
    output_dir=Path(OUTPUT_DIR),
    delay=0.3,
    batch_pause=2.0,
)

print('\nTranslated question files:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    print(f'  {f.name}')

In [ ]:
# Show sample translations
import json

with open(os.path.join(OUTPUT_DIR, 'dev.json'), encoding='utf-8') as f:
    dev = json.load(f)

for entry in dev[:10]:
    print(f"EN: {entry.get('question_en', entry.get('question', ''))}")
    print(f"ID: {entry.get('question_id', '')}")
    print()

### Next Steps

1. Download `translated_questions/` from `/kaggle/working/`
2. Place in `idspider1/data/intermediate/translated_questions/`
3. Run Phase 2 locally: `python -m src_new.run_pipeline --phase 2`
4. Run Phase 3 locally: `python -m src_new.run_pipeline --phase 3`
5. Run verification: `python -m src_new.run_pipeline --phase verify`